# <a href="https://girafe.ai/" target="_blank" rel="noopener noreferrer"><img src="https://raw.githubusercontent.com/girafe-ai/ml-course/7096a5df4cada5ee651be1e3215c2f7fb8a7e0bf/logo_margin.svg" alt="girafe-ai logo" width="150px" align="left"></a> YMLS Uzbekistan, ML Fundamentals, fall 2026 <a class="tocSkip">

# Week 4 seminar. Linear classification and logistic regression

Forty flats in Tashkent and one question: will a flat sell within 30 days? Least squares on the labels goes first and breaks. Then logistic regression from scratch in NumPy, its weights read as odds, the penalty `C`, curved boundaries chosen by cross-validation, and more than two classes. At the end, real data: tumours and handwritten digits.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girafe-ai/ml-course/blob/26f_ymls_uz/week0_04_logistic_regression/week0_04_01_logistic_regression.ipynb)

This week's [lecture slides](https://github.com/girafe-ai/ml-course/blob/26f_ymls_uz/week0_04_logistic_regression/ml-basic_lect04_logistic_regression.pdf) are in the same folder. The links to the interactive pages are in the course chat.

_YMLS Uzbekistan, ML Fundamentals, fall 2026_

Fill in the cells marked `# YOUR CODE HERE`. The cell after each exercise checks your answer and says what is wrong. Run the cells from top to bottom, since later sections use earlier variables. It takes about 80 minutes.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler


SEED = 42  # one seed for every split and random generator

# Plot style: gray axes, colour only for data
INK, INK2, INK3, LINE = "#1D1D1F", "#6E6E73", "#86868B", "#D2D2D7"
ACCENT, ORANGE, GREEN, PURPLE = "#0071E3", "#FF9F0A", "#34C759", "#AF52DE"
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"],
        "font.size": 13,
        "mathtext.fontset": "cm",
        "axes.edgecolor": LINE,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "axes.prop_cycle": plt.cycler(color=[ACCENT, ORANGE, GREEN, PURPLE, INK3]),
        "xtick.color": INK3,
        "ytick.color": INK3,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "xtick.minor.size": 0,
        "ytick.minor.size": 0,
        "legend.frameon": False,
        "legend.fontsize": 11,
        "lines.linewidth": 2.2,
        "lines.markersize": 6,
        "figure.facecolor": "white",
    }
)
# shading for a predicted probability: orange near 0, white at 0.5, blue near 1
PROB_CMAP = LinearSegmentedColormap.from_list("probability", [ORANGE, "white", ACCENT])

## 1. Will the flat sell?

Forty flats are listed for sale in Tashkent. Each has a markup, the asking price against the market estimate in percent ($+10$ is 10% above the estimate, $-5$ is 5% below), and a distance to the nearest metro station in km. The label `sold` is 1 if the flat sold within 30 days and 0 if it did not. The data is synthetic, and section 3 compares the fitted weights with the ones that generated it. We also draw 400 new flats from the same market to see how a model does on flats it has not seen.

In [ ]:
def sold_sample(seed, n):
    """n flats: the markup in percent, the distance to the metro in km, sold (1) or not (0)."""
    rng = np.random.default_rng(seed)
    markup = np.round(rng.uniform(-15, 25, n), 0)  # asking price against the estimate, %
    metro = np.round(rng.uniform(0.2, 4.0, n), 1)  # km to the nearest metro station
    log_odds = 2.0 - 0.20 * markup - 0.8 * metro
    sold = (rng.uniform(size=n) < 1 / (1 + np.exp(-log_odds))).astype(int)
    return markup, metro, sold


def scatter_sold(ax, x, y, sold, size=45):
    """Flats on a plane: sold ones as blue circles, the others as orange squares."""
    style = {"s": size, "edgecolor": "white", "linewidth": 0.8, "zorder": 3}
    ax.scatter(x[sold == 1], y[sold == 1], color=ACCENT, label="sold", **style)
    ax.scatter(x[sold == 0], y[sold == 0], color=ORANGE, marker="s", label="not sold", **style)


markup, metro, sold = sold_sample(24, 40)  # the 40 flats we train on
markup_new, metro_new, sold_new = sold_sample(5024, 400)  # 400 new flats from the same market

print("markup, %:", markup[:8].astype(int), "...")
print("metro, km:", metro[:8], "...")
print("sold:     ", sold[:8], "...")
print(
    f"{sold.sum()} of the {len(sold)} flats sold, {sold_new.mean():.1%} of the 400 new flats sold"
)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
scatter_sold(ax, metro, markup, sold)
ax.set(xlabel="distance to the metro, km", ylabel="markup, %")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1), ncols=3)
plt.show()

### 1.1 The baseline

As in week 1, start with a model that ignores the features. 19 of the 40 flats sold, so the most frequent answer is "not sold".

**Exercise 1.1.** Compute the accuracy of the model that always answers "not sold": `baseline_acc` on the 40 flats and `baseline_acc_new` on the 400 new flats. Accuracy is the share of right answers.

In [ ]:
# YOUR CODE HERE: the share of flats with sold == 0, in sold and in sold_new
baseline_acc = None
baseline_acc_new = None

In [ ]:
assert baseline_acc is not None, "Fill in baseline_acc and baseline_acc_new above"
assert np.isclose(baseline_acc, 0.525), "On the 40 flats the baseline is right 21 times: 0.525"
assert np.isclose(baseline_acc_new, 0.565), "On the 400 new flats the baseline should get 0.565"
print(
    f'always "not sold": accuracy {baseline_acc:.3f} on the 40 flats, '
    f"{baseline_acc_new:.3f} on the 400 new flats"
)

### 1.2 Least squares on the labels

The labels are the numbers 0 and 1, so we can fit the linear regression of week 2 to them and answer "sold" when the line is above 0.5. Use the markup alone.

In [ ]:
line = LinearRegression().fit(markup[:, None], sold)  # least squares on the labels 0 and 1
line_threshold = (0.5 - line.intercept_) / line.coef_[0]  # the markup where the line crosses 0.5
line_acc = np.mean((line.predict(markup[:, None]) >= 0.5) == sold)
print(f"the line: {line.intercept_:.3f} - {-line.coef_[0]:.4f} markup")
print(
    f"its answer at a 15% discount: {line.predict([[-15]])[0]:.2f}, "
    f"at a 25% markup: {line.predict([[25]])[0]:.2f}"
)
print(
    f"it crosses 0.5 at a markup of {line_threshold:.2f}%, accuracy on the 40 flats {line_acc:.3f}"
)

cheap = np.array([-60.0, -65, -70, -75, -80])  # five flats far below the market, all sold
markup_45, sold_45 = np.r_[markup, cheap], np.r_[sold, np.ones(5, dtype=int)]
line_45 = LinearRegression().fit(markup_45[:, None], sold_45)
line_45_threshold = (0.5 - line_45.intercept_) / line_45.coef_[0]
line_45_acc = np.mean((line_45.predict(markup[:, None]) >= 0.5) == sold)
print(
    f"with the five cheap flats: it crosses 0.5 at {line_45_threshold:.2f}%, "
    f"accuracy on the same 40 flats {line_45_acc:.3f}"
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
panels = [
    ("The 40 flats", markup, sold, (-17, 27)),
    ("With five very cheap flats", markup_45, sold_45, (-85, 27)),
]
for ax, (title, x, y, x_range) in zip(axes, panels, strict=True):
    grid = np.array(x_range, dtype=float)
    for level in (0, 0.5, 1):
        ax.axhline(level, color=LINE, lw=0.8)
    scatter_sold(ax, x, y, y)
    ax.plot(
        grid,
        line.predict(grid[:, None]),
        color=INK3 if len(x) > 40 else INK,
        label="least squares on the 40 flats",
    )
    ax.axvline(line_threshold, color=INK3, ls="--", lw=1.2)
    if len(x) > 40:
        ax.plot(grid, line_45.predict(grid[:, None]), color=INK, label="least squares on all 45")
        ax.axvline(line_45_threshold, color=INK, ls="--", lw=1.2)
    ax.set(xlabel="markup, %", xlim=x_range, ylim=(-0.35, 1.9), title=title)
axes[0].set_ylabel("label and the line's answer")
axes[1].legend(loc="lower left")
fig.tight_layout()
plt.show()

On the 40 flats the line looks fine: it crosses 0.5 at a markup of 3.88% and gets 0.800 of the flats right. But its answers leave the range from 0 to 1 (1.06 at a 15% discount and $-0.13$ at a 25% markup), so they cannot be read as probabilities. The line is also fragile. Add five flats priced 60 to 80% below the market, all of which sold. The old line already had them on the "sold" side, yet the new line tilts towards them: its threshold moves from 3.88% to $-0.40\%$, and its accuracy on the original 40 flats falls to 0.750. Squared error charges the line for being far from 1 on those five flats, although its answer for them was right. The page `01_odds_and_sigmoid.html` has both switches.

## 2. Logistic regression by hand

We want a probability $p$ that a flat sells. A probability lies between 0 and 1, while a linear function $z = b + \mathbf{w}^\top \mathbf{x}$ takes any value. The odds $p / (1 - p)$ run from 0 to infinity, and their logarithm, the log-odds, covers the whole line. Logistic regression models the log-odds with a linear function and solves for $p$:

$$\log \frac{p}{1 - p} = z \quad \Longrightarrow \quad p = \sigma(z) = \frac{1}{1 + e^{-z}}.$$

The function $\sigma$ is the sigmoid.

### 2.1 The sigmoid

**Exercise 2.1.** Write `sigmoid(z)`. It should work for a number and for an array.

In [ ]:
def sigmoid(z):
    """The sigmoid 1 / (1 + exp(-z)) of a number or of every element of an array."""
    # YOUR CODE HERE: one line with np.exp
    return None

In [ ]:
assert sigmoid(0.0) is not None, "Fill in sigmoid above"
z_check = np.array([-3.0, -1.0, 0.5, 4.0])
assert np.isclose(sigmoid(0.0), 0.5), "sigmoid(0) should be 0.5"
assert np.allclose(sigmoid(z_check), 1 / (1 + np.exp(-z_check))), "Expect 1 / (1 + exp(-z))"
assert np.allclose(sigmoid(-z_check), 1 - sigmoid(z_check)), "Expect sigmoid(-z) = 1 - sigmoid(z)"
print("z:         ", z_check)
print("sigmoid(z):", sigmoid(z_check).round(3))

z_grid = np.linspace(-7, 7, 300)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.axhline(0.5, color=LINE, lw=0.8)
ax.axvline(0, color=LINE, lw=0.8)
ax.plot(z_grid, sigmoid(z_grid), color=ACCENT)
ax.set(xlabel="log-odds $z$", ylabel=r"probability $\sigma(z)$", ylim=(-0.03, 1.03))
plt.show()

The sigmoid is 0.5 at $z = 0$, close to 1 for a large positive $z$ and close to 0 for a large negative one. The check also used its symmetry, $\sigma(-z) = 1 - \sigma(z)$.

### 2.2 The log-loss

Week 2 gave the recipe: choose a distribution for the target, and its negative log-likelihood is the loss. Here the target is 0 or 1, a Bernoulli variable with $P(y_i = 1) = p_i$. Its negative log-likelihood, divided by the number of flats $n$, is the log-loss:

$$L(\mathbf{w}) = -\frac{1}{n} \sum_{i=1}^{n} \bigl[ y_i \log p_i + (1 - y_i) \log (1 - p_i) \bigr], \qquad p_i = \sigma(z_i).$$

A flat that sold ($y_i = 1$) costs $-\log p_i$, and a flat that did not costs $-\log (1 - p_i)$.

**Exercise 2.2.** Write `mean_log_loss(y, p)` by this formula. The first line of the function is given: it keeps $p$ away from 0 and 1, where the logarithm is infinite.

In [ ]:
def mean_log_loss(y, p):
    """Mean log-loss of the labels y (0 or 1) under the predicted probabilities p."""
    p = np.clip(p, 1e-15, 1 - 1e-15)
    # YOUR CODE HERE: minus the mean of y * log(p) + (1 - y) * log(1 - p)
    return None

In [ ]:
half = np.full(40, 0.5)
assert mean_log_loss(sold, half) is not None, "Fill in mean_log_loss above"
assert np.isclose(mean_log_loss(sold, half), np.log(2)), "With p = 0.5 everywhere expect log 2"
p_check = np.random.default_rng(SEED).uniform(0.05, 0.95, 40)
assert np.isclose(
    mean_log_loss(sold, p_check), log_loss(sold, p_check)
), "The result differs from sklearn.metrics.log_loss: check the signs and the mean"
print(f"0.5 for every flat:   {mean_log_loss(sold, half):.3f}")
print(f"{sold.mean():.3f} for every flat: {mean_log_loss(sold, np.full(40, sold.mean())):.3f}")
print("the cost of one flat that sold, by the probability the model gave it:")
for p in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"  p = {p:4.2f}: {mean_log_loss(np.array([1]), np.array([p])):.2f}")

A model that says 0.5 for every flat scores $\log 2 = 0.693$, and one that says 0.475, the share of flats that sold, scores 0.692. A useful model has to get below that. The cost of a single flat has no upper limit: 0.01 when a sold flat was predicted at 0.99, and 4.61 when it was predicted at 0.01. `sklearn.metrics.log_loss` computes the same mean. The page `02_losses_and_margin.html` puts this loss next to the others from the lecture.

### 2.3 The gradient

Gradient descent needs features on one scale (week 2), so standardise the two features first. Then put a column of ones in front of them, which makes the intercept the first weight. Call this matrix $X$, the weights $\mathbf{w} = (b, w_1, w_2)$ and the predictions $\mathbf{p} = \sigma(X \mathbf{w})$.

In [ ]:
X = np.column_stack([markup, metro])  # the raw features: percent and km
scaler = StandardScaler().fit(X)
Z = scaler.transform(X)  # every column now has mean 0 and standard deviation 1
Z1 = np.column_stack([np.ones(len(Z)), Z])  # a column of ones for the intercept
print("means:              ", scaler.mean_.round(3))
print("standard deviations:", scaler.scale_.round(3))
print("the first three rows of the matrix:")
print(Z1[:3].round(3))

The gradient of the log-loss has the same shape as in linear regression, the features times the errors:

$$\nabla L(\mathbf{w}) = \frac{1}{n} X^\top (\mathbf{p} - \mathbf{y}).$$

The lecture derived $X^\top (\mathbf{p} - \mathbf{y})$ for the sum of the losses. Our loss is the mean, hence the division by $n$.

**Exercise 2.3.** Write `gradient(X, y, w)`.

In [ ]:
def gradient(X, y, w):
    """Gradient of the mean log-loss at the weights w: X^T (p - y) / n."""
    # YOUR CODE HERE: p = sigmoid(X @ w), then the formula
    return None

In [ ]:
assert sigmoid(0.0) is not None, "Finish Exercise 2.1 first"
assert mean_log_loss(sold, np.full(40, 0.5)) is not None, "Finish Exercise 2.2 first"
grad_zero = gradient(Z1, sold, np.zeros(3))
assert grad_zero is not None, "Fill in gradient above"
assert grad_zero.shape == (3,), "Expect one number per weight: shape (3,)"

# the slope of the loss measured by moving one weight at a time
w_check, shift = np.array([0.3, -1.0, 0.5]), 1e-6
numeric = np.array(
    [
        (
            mean_log_loss(sold, sigmoid(Z1 @ (w_check + shift * unit)))
            - mean_log_loss(sold, sigmoid(Z1 @ (w_check - shift * unit)))
        )
        / (2 * shift)
        for unit in np.eye(3)
    ]
)
assert np.allclose(
    gradient(Z1, sold, w_check), numeric, atol=1e-6
), "The gradient differs from the measured slope of the loss: X^T (p - y) / n"
print("gradient at zero weights:", grad_zero.round(3))

### 2.4 Gradient descent

A zero gradient means $X^\top (\sigma(X \mathbf{w}) - \mathbf{y}) = 0$, and this equation has no closed-form solution. The loss is convex, so the gradient descent of week 2 finds its only minimum:

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \, \nabla L(\mathbf{w}).$$

**Exercise 2.4.** Fill in the step inside `gradient_descent`. The function starts from zero weights and returns the weights after every step, the start included.

In [ ]:
def gradient_descent(X, y, eta, steps):
    """Gradient descent from zero weights. Returns an array with one row of weights per step."""
    w = np.zeros(X.shape[1])
    path = [w]
    for _ in range(steps):
        # YOUR CODE HERE: one step, w = w - eta * (the gradient at w), then path.append(w)
        pass
    return np.array(path)  # shape (steps + 1, number of weights), row 0 is the start

In [ ]:
assert gradient(Z1, sold, np.zeros(3)) is not None, "Finish Exercise 2.3 first"
gd_path = gradient_descent(Z1, sold, 1.0, 100)
assert gd_path.shape == (101, 3), "Fill in the loop in gradient_descent above: one row per step"
assert np.allclose(
    gd_path[1], -gradient(Z1, sold, np.zeros(3))
), "With eta = 1 the first step from zero should be minus the gradient"
assert not np.allclose(
    gd_path[1], gd_path[-1]
), "Every row is the same: write w = w - ..., since w -= ... changes the stored rows too"
gd_loss = mean_log_loss(sold, sigmoid(Z1 @ gd_path[-1]))
assert np.isclose(
    gd_loss, 0.3971, atol=2e-4
), "After 100 steps with eta = 1 expect a loss of 0.3971"
print(f"loss at the start: {mean_log_loss(sold, sigmoid(Z1 @ gd_path[0])):.4f}")
print(f"after 100 steps:   {gd_loss:.4f}, weights {gd_path[-1].round(3)}")

Now the learning rate. Run 1,000 steps with four values of $\eta$ and count the steps until the loss is within 0.001 of its minimum.

In [ ]:
assert gradient_descent(Z1, sold, 1.0, 2).shape == (3, 3), "Finish Exercise 2.4 first"


def loss_curve(path):
    """The mean log-loss on the 40 flats after every step of a path."""
    return np.array([mean_log_loss(sold, sigmoid(Z1 @ w)) for w in path])


w_min = gradient_descent(Z1, sold, 5.0, 1000)[-1]  # long enough to sit at the minimum
loss_min = mean_log_loss(sold, sigmoid(Z1 @ w_min))
print(f"the minimum of the loss: {loss_min:.4f} at the weights {w_min.round(3)}\n")

curves, steps_needed = {}, {}
print("  eta   loss after 10 steps   after 100 steps   steps to get within 0.001")
for eta in [0.1, 1.0, 5.0, 20.0]:
    curves[eta] = loss_curve(gradient_descent(Z1, sold, eta, 1000))
    close = np.nonzero(curves[eta] <= loss_min + 0.001)[0]
    steps_needed[eta] = int(close[0]) if len(close) else None
    count = "never" if steps_needed[eta] is None else steps_needed[eta]
    print(f"{eta:5g} {curves[eta][10]:21.4f} {curves[eta][100]:17.4f} {count:>26}")

fig, ax = plt.subplots(figsize=(8.5, 4.5))
step_numbers = np.arange(1, 1001)
for eta, color in zip(curves, [ACCENT, ORANGE, GREEN, INK3], strict=True):
    ax.plot(
        step_numbers,
        curves[eta][1:],
        color=color,
        lw=1.2 if eta == 20 else 2.2,
        label=rf"$\eta$ = {eta:g}",
    )
    if steps_needed[eta] is not None:
        ax.scatter(
            [steps_needed[eta]],
            [curves[eta][steps_needed[eta]]],
            s=60,
            color=color,
            edgecolor="white",
            zorder=4,
        )
ax.axhline(loss_min, color=INK3, ls="--", lw=1.2)
ax.annotate(
    "the minimum",
    (1, loss_min),
    xytext=(0, -14),
    textcoords="offset points",
    fontsize=11,
    color=INK2,
)
ax.set(xscale="log", xlabel="step", ylabel="log-loss on the 40 flats", ylim=(0.35, 0.95))
ax.legend(loc="upper right", ncols=2)
plt.show()

Every run starts at $\log 2 = 0.693$, because zero weights give $p = 0.5$ for every flat. The minimum of the loss is 0.397. With $\eta = 0.1$ the descent needs 441 steps to get within 0.001 of it, with $\eta = 1$ it needs 43, and with $\eta = 5$ only 7. A larger step is faster only up to a point: at $\eta = 20$ every step jumps over the minimum, and after 1,000 steps the loss is still 0.68. On the page `03_train_logistic.html` you can run the same descent one step at a time.

### 2.5 The same with scikit-learn

`LogisticRegression` minimises the same loss with a faster optimiser. By default it also adds a penalty on the weights, which section 4 explains. `C=np.inf` switches the penalty off.

In [ ]:
assert gradient_descent(Z1, sold, 1.0, 2).shape == (3, 3), "Finish Exercise 2.4 first"
sk_model = LogisticRegression(C=np.inf).fit(Z, sold)  # the column of ones is not needed here
sk_weights = np.r_[sk_model.intercept_, sk_model.coef_[0]]
print("gradient descent, 1,000 steps:", gradient_descent(Z1, sold, 1.0, 1000)[-1].round(3))
print("scikit-learn:                 ", sk_weights.round(3))

The two answers agree to three decimals: an intercept of $-0.439$, a weight of $-2.290$ for the markup and $-1.013$ for the metro, each per standard deviation of its feature.

### 2.6 Newton's method (optional)

Gradient descent looks only at the slope of the loss. Newton's method also uses its curvature, the Hessian $H = \frac{1}{n} X^\top S X$ with $S = \mathrm{diag}\bigl(p_i (1 - p_i)\bigr)$. Each step goes to the minimum of the parabola that fits the loss at the current weights:

$$\mathbf{w} \leftarrow \mathbf{w} - H^{-1} \nabla L(\mathbf{w}).$$

**Exercise 2.6 (optional).** Write `newton_step(X, y, w)`. Solve the system with `np.linalg.solve`, as in week 2, without computing the inverse. Nothing later depends on this exercise.

In [ ]:
def newton_step(X, y, w):
    """One step of Newton's method on the mean log-loss: w - H^{-1} g."""
    # YOUR CODE HERE (optional): p = sigmoid(X @ w); the Hessian is X^T S X / n, where
    # S X is X with row i multiplied by p_i * (1 - p_i); return w - solve(H, gradient)
    return None

In [ ]:
assert gradient_descent(Z1, sold, 1.0, 2).shape == (3, 3), "Finish Exercise 2.4 first"
if newton_step(Z1, sold, np.zeros(3)) is None:
    print("Exercise 2.6 is optional and is not filled in. Nothing below depends on it.")
else:
    w_newton, newton_steps = np.zeros(3), 0
    print("step   log-loss")
    while np.linalg.norm(gradient(Z1, sold, w_newton)) > 1e-10 and newton_steps < 50:
        w_newton = newton_step(Z1, sold, w_newton)
        newton_steps += 1
        print(f"{newton_steps:4d}   {mean_log_loss(sold, sigmoid(Z1 @ w_newton)):.10f}")
    assert newton_steps == 6, "Expect 6 steps: check the Hessian X^T S X / n and the sign"
    print(f"the gradient is zero after {newton_steps} steps, weights {w_newton.round(3)}")

Newton's method brings the gradient to zero in 6 steps, where gradient descent with $\eta = 1$ needed 43 to get within 0.001 of the minimum. Each of its steps costs more, since it builds the Hessian and solves a linear system. Lecture 5 returns to optimisation methods.

## 3. Reading the model

From here on the models come from scikit-learn, so this section runs even if section 2 is unfinished.

### 3.1 One feature

Start with the markup alone, in percent and without standardising, so that the weight reads per percent of markup.

In [ ]:
one = LogisticRegression(C=np.inf).fit(markup[:, None], sold)
b1, w1 = one.intercept_[0], one.coef_[0, 0]
print(f"log-odds = {b1:.3f} - {-w1:.3f} markup, so p = 0.5 at a markup of {-b1 / w1:.2f}%\n")
print("markup   probability   odds   log-odds")
for value in [-10, 0, 4, 10, 20]:
    p = one.predict_proba([[value]])[0, 1]
    print(f"{value:5d}% {p:13.3f} {p / (1 - p):6.3g} {np.log(p / (1 - p)):10.2f}")

one_45 = LogisticRegression(C=np.inf).fit(markup_45[:, None], sold_45)
one_45_threshold = -one_45.intercept_[0] / one_45.coef_[0, 0]
print(f"\nwith the five cheap flats: p = 0.5 at a markup of {one_45_threshold:.2f}%")

markup_grid = np.linspace(-17, 27, 300)
fig, ax = plt.subplots(figsize=(8, 4.2))
for level in (0, 0.5, 1):
    ax.axhline(level, color=LINE, lw=0.8)
scatter_sold(ax, markup, sold, sold)
ax.plot(markup_grid, line.predict(markup_grid[:, None]), color=INK3, lw=1.5, label="least squares")
ax.plot(
    markup_grid,
    one.predict_proba(markup_grid[:, None])[:, 1],
    color=INK,
    label="logistic regression",
)
ax.axvline(-b1 / w1, color=INK, ls="--", lw=1.2)
ax.set(xlabel="markup, %", ylabel="label and predicted probability", ylim=(-0.2, 1.2))
ax.legend(loc="upper right")
plt.show()

The model is $\text{log-odds} = 0.682 - 0.173 \cdot \text{markup}$. The log-odds are a straight line in the markup, and the probability is an S-curve that passes 0.5 at a markup of 3.95%. A flat at a 10% discount sells with probability 0.918, which is odds of 11.1 to 1. At a 20% markup the probability is 0.059.

The five very cheap flats that broke least squares leave this threshold at 3.95%. The log-loss charges almost nothing for a flat that the model already gets right with confidence, so those flats do not pull on the curve.

### 3.2 Two features and odds ratios

Now both features, again in their own units.

In [ ]:
model = LogisticRegression(C=np.inf).fit(X, sold)  # X: the markup in percent, the metro in km
b, (w_markup, w_metro) = model.intercept_[0], model.coef_[0]
print(f"log-odds = {b:.2f} - {-w_markup:.3f} markup - {-w_metro:.3f} metro")

The fitted model is $\text{log-odds} = 2.59 - 0.213 \cdot \text{markup} - 0.983 \cdot \text{metro}$. The data was generated with $2.0 - 0.20 \cdot \text{markup} - 0.8 \cdot \text{metro}$, so 40 flats recover the signs and the rough sizes. The weights are the standardised ones of section 2 divided by the standard deviations: $-2.290 / 10.775 = -0.213$ per percent of markup.

A weight is the change in the log-odds when its feature grows by one unit and the other feature stays fixed. Adding $w$ to the log-odds multiplies the odds by $e^{w}$. This number is called the odds ratio.

**Exercise 3.1.** Compute `odds_ratio_markup` and `odds_ratio_metro` from `w_markup` and `w_metro`.

In [ ]:
# YOUR CODE HERE: np.exp of each weight
odds_ratio_markup = None
odds_ratio_metro = None

In [ ]:
assert odds_ratio_markup is not None, "Fill in the two odds ratios above"
assert np.isclose(odds_ratio_markup, 0.809, atol=2e-3), "Expect exp(w_markup) = 0.809"
assert np.isclose(odds_ratio_metro, 0.374, atol=2e-3), "Expect exp(w_metro) = 0.374"
print(f"one more percent of markup multiplies the odds by {odds_ratio_markup:.3f}")
print(f"one more km to the metro multiplies the odds by   {odds_ratio_metro:.3f}\n")

p_near, p_far = model.predict_proba([[5, 1.0], [5, 2.0]])[:, 1]  # a 5% markup, 1 km and 2 km
odds_near, odds_far = p_near / (1 - p_near), p_far / (1 - p_far)
print(f"a flat with a 5% markup, 1 km from the metro: p = {p_near:.3f}, odds {odds_near:.3f}")
print(f"the same flat 2 km from the metro:            p = {p_far:.3f}, odds {odds_far:.3f}")
print(f"the ratio of the two odds: {odds_far / odds_near:.3f}")

Each extra percent of markup multiplies the odds of selling by 0.809, and each extra kilometre to the metro by 0.374. The odds ratio is the same for every flat. The change in probability depends on where the flat starts: the extra kilometre takes the flat above from 0.633 to 0.392, and it would hardly move a flat whose probability is already close to 0 or to 1.

### 3.3 The boundary

The model answers "sold" when $p \ge 0.5$, that is when the log-odds are at least 0. On the plane of the two features the points with log-odds of exactly 0 form a straight line, the decision boundary:

$$b + w_1 \cdot \text{markup} + w_2 \cdot \text{metro} = 0 \quad \Longrightarrow \quad \text{markup} = -\frac{b + w_2 \cdot \text{metro}}{w_1}.$$

In [ ]:
def boundary_markup(metro_km):
    """The markup at which the model says exactly 0.5, for a flat this far from the metro."""
    return -(b + w_metro * metro_km) / w_markup


def grid_points(x_range, y_range, n=200):
    """A fine grid over a plane: the two coordinate matrices and the points as rows (x, y)."""
    xx, yy = np.meshgrid(np.linspace(*x_range, n), np.linspace(*y_range, n))
    return xx, yy, np.column_stack([xx.ravel(), yy.ravel()])


print(
    f"the boundary: a markup of {boundary_markup(0.5):.1f}% at 0.5 km from the metro, "
    f"{boundary_markup(3.0):.1f}% at 3 km"
)

xx, yy, points = grid_points((0, 4.2), (-17, 27))  # x is the metro, y is the markup
p_grid = model.predict_proba(points[:, ::-1])[:, 1].reshape(
    xx.shape
)  # the model wants markup first
fig, ax = plt.subplots(figsize=(8, 5))
shade = ax.contourf(xx, yy, p_grid, levels=np.linspace(0, 1, 11), cmap=PROB_CMAP, alpha=0.5)
scatter_sold(ax, metro, markup, sold)
metro_grid = np.array([0, 4.2])
ax.plot(metro_grid, boundary_markup(metro_grid), color=INK, label="p = 0.5")
ax.set(xlabel="distance to the metro, km", ylabel="markup, %", ylim=(-17, 27))
ax.legend(loc="lower left", bbox_to_anchor=(0, 1), ncols=3)
fig.colorbar(shade, ax=ax, label="predicted probability of selling")
plt.show()

A flat 0.5 km from the metro is predicted to sell up to a markup of 9.9%. At 3 km the boundary is at $-1.7\%$, so such a flat needs a discount. The shading is the probability, and it changes smoothly across the line. The threshold 0.5 is a choice, and lecture 5 shows what moving it does.

### 3.4 How good is it

Two numbers for each model: the accuracy at the threshold 0.5, and the log-loss, which also looks at how sure the model was.

In [ ]:
X_new = np.column_stack([markup_new, metro_new])
score_rows = {}


def add_scores(name, p_train, p_test):
    """Accuracy at the threshold 0.5 and log-loss, on the 40 flats and on the 400 new ones."""
    score_rows[name] = (
        np.mean((p_train >= 0.5) == sold),
        np.mean((p_test >= 0.5) == sold_new),
        log_loss(sold, p_train),
        log_loss(sold_new, p_test),
    )


share = sold.mean()  # the baseline says this probability for every flat: 0.475, "not sold"
add_scores('always "not sold"', np.full(40, share), np.full(400, share))
add_scores(
    "markup only",
    one.predict_proba(markup[:, None])[:, 1],
    one.predict_proba(markup_new[:, None])[:, 1],
)
add_scores("markup and metro", model.predict_proba(X)[:, 1], model.predict_proba(X_new)[:, 1])

print("                    accuracy             log-loss")
print("                    40 flats   400 new   40 flats   400 new")
for name, row in score_rows.items():
    print(f"{name:18s} {row[0]:9.3f} {row[1]:9.3f} {row[2]:10.3f} {row[3]:9.3f}")

Both models beat the baseline on the new flats: 0.820 for the markup alone and 0.843 with the metro, against 0.565. The log-loss falls from 0.688 to 0.390 and 0.352. A model with two or three parameters has little room to overfit 40 flats, and here it scores no worse on the new flats than on the ones it was trained on.

### 3.5 Can the probabilities be trusted

If the model says 0.7 for a group of flats, about 70% of them should sell. Check this on the 400 new flats: cut them into five bins by the predicted probability and compare the mean prediction in each bin with the share that sold.

**Exercise 3.2.** Fill in the loop. For each bin store the number of flats, the mean of `p_new` and the mean of `sold_new` among the flats of the bin.

In [ ]:
p_new = model.predict_proba(X_new)[:, 1]  # the probability of selling for each new flat
bin_of = np.digitize(p_new, [0.2, 0.4, 0.6, 0.8])  # 0 for p below 0.2, ..., 4 for p from 0.8
flats_in_bin, mean_predicted, share_sold = [], [], []
for k in range(5):
    inside = bin_of == k  # True for the flats of bin k
    # YOUR CODE HERE: append inside.sum(), the mean of p_new[inside] and the mean of
    # sold_new[inside] to the three lists

In [ ]:
assert len(flats_in_bin) == 5, "Fill in the loop above: five values in each of the three lists"
assert list(flats_in_bin) == [
    158,
    51,
    34,
    41,
    116,
], "Expect 158, 51, 34, 41 and 116 flats in the bins"
assert np.allclose(
    mean_predicted, [0.061, 0.286, 0.498, 0.699, 0.922], atol=2e-3
), "The mean predictions should be 0.061, 0.286, 0.498, 0.699, 0.922"
assert np.allclose(
    share_sold, [0.070, 0.255, 0.441, 0.634, 0.940], atol=2e-3
), "The shares that sold should be 0.070, 0.255, 0.441, 0.634, 0.940"
print("predicted    flats   mean predicted   share that sold")
for k in range(5):
    print(
        f"{0.2 * k:.1f} to {0.2 * k + 0.2:.1f} {flats_in_bin[k]:7d} {mean_predicted[k]:16.2f} "
        f"{share_sold[k]:17.2f}"
    )

fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot([0, 1], [0, 1], color=INK3, ls="--", lw=1.2, label="perfect calibration")
ax.plot(mean_predicted, share_sold, "o-", color=ACCENT, label="the model, 400 new flats")
ax.set(xlabel="mean predicted probability", ylabel="share that sold", aspect="equal")
ax.legend(loc="upper left")
plt.show()

The two columns stay close. Among the 41 new flats with a prediction between 0.6 and 0.8 the mean prediction is 0.70 and 63% sold. In the lowest bin, 158 flats, the model says 0.06 on average and 7% sold. A model like this is called well calibrated. Logistic regression usually is, because the log-loss rewards honest probabilities.

Accuracy is not the whole story. It treats both kinds of mistake as equal and it depends on the threshold. Lecture 5 is about the metrics that look at this.

## 4. A perfect split and the penalty

Twelve flats: the six with markups from $-12\%$ to 1% sold and the six from 6% to 21% did not. Any threshold between 1% and 6% splits them without a mistake.

In [ ]:
sep_markup = np.array([-12, -9, -7, -4, -2, 1, 6, 8, 11, 14, 17, 21], dtype=float)
sep_sold = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
x_sep = (sep_markup - sep_markup.mean()) / sep_markup.std()  # the standardised markup
X_sep = np.column_stack([np.ones(12), x_sep])  # with a column of ones, for our gradient descent
sep_grid = np.linspace(-14, 23, 400)  # markups to draw curves on
x_sep_grid = (sep_grid - sep_markup.mean()) / sep_markup.std()
print(f"mean markup {sep_markup.mean():.3f}%, standard deviation {sep_markup.std():.3f}")

Run the gradient descent of section 2 on them for 100,000 steps, with the standardised markup as the only feature and $\eta = 1$.

In [ ]:
assert gradient_descent(Z1, sold, 1.0, 2).shape == (3, 3), "Finish Exercise 2.4 first"
sep_path = gradient_descent(X_sep, sep_sold, 1.0, 100_000)
print("  steps    slope   log-loss")
for steps in [10, 100, 1000, 10_000, 100_000]:
    loss = mean_log_loss(sep_sold, sigmoid(X_sep @ sep_path[steps]))
    print(f"{steps:7,d} {sep_path[steps, 1]:8.2f} {loss:10.5f}")

fig, ax = plt.subplots(figsize=(8, 4.2))
scatter_sold(ax, sep_markup, sep_sold, sep_sold)
for steps, color in zip([10, 100, 1000, 10_000], [LINE, INK3, INK2, INK], strict=True):
    b_t, w_t = sep_path[steps]
    ax.plot(sep_grid, sigmoid(b_t + w_t * x_sep_grid), color=color, label=f"{steps:,} steps")
ax.set(xlabel="markup, %", ylabel="predicted probability", title="Gradient descent, no penalty")
ax.legend(loc="upper right")
plt.show()

This descent does not converge. The slope is $-2.10$ after 10 steps, $-5.28$ after 100, $-10.97$ after 1,000, $-19.23$ after 10,000 and $-28.50$ after 100,000, and the loss keeps creeping towards 0. Once the classes are split, doubling every weight pushes every probability further towards the right answer and lowers the loss, so no finite weights are the best ones. The S-curve turns into a step, and the model claims to be certain about a flat just to either side of the gap.

The remedy is the one from week 3, a penalty on the size of the weights. `LogisticRegression(C=...)` minimises

$$\frac{1}{2} \lVert \mathbf{w} \rVert_2^2 + C \sum_{i=1}^{n} \ell_i,$$

where $\ell_i$ is the log-loss of flat $i$. In the form "loss $+ \lambda \lVert \mathbf{w} \rVert_2^2$" this is ridge again with $\lambda = 1 / (2C)$, so a small $C$ is a strong penalty. The intercept is not penalised. The default is `C=1.0`, and `C=np.inf` is no penalty at all.

**Exercise 4.1.** For every `C` in `C_VALUES`, fit `LogisticRegression(C=C)` on the standardised markup `x_sep[:, None]` and append the fitted model to `sep_models`.

In [ ]:
C_VALUES = [0.1, 1, 10]
sep_models = []
# YOUR CODE HERE: for every C in C_VALUES, fit LogisticRegression(C=C) on x_sep[:, None],
# sep_sold and append the fitted model to sep_models

In [ ]:
assert len(sep_models) == 3, "Fill in the loop above: one fitted model per C"
sep_slopes = [fit.coef_[0, 0] for fit in sep_models]
assert np.allclose(
    sep_slopes, [-0.41, -1.65, -3.96], atol=0.01
), "Expect the slopes -0.41, -1.65 and -3.96: fit on x_sep[:, None] and sep_sold"
for C, slope in zip(C_VALUES, sep_slopes, strict=True):
    print(f"C = {C:4g}: slope {slope:.2f}")

fig, ax = plt.subplots(figsize=(8, 4.2))
scatter_sold(ax, sep_markup, sep_sold, sep_sold)
for C, fit, color in zip(C_VALUES, sep_models, [GREEN, PURPLE, INK], strict=True):
    ax.plot(sep_grid, fit.predict_proba(x_sep_grid[:, None])[:, 1], color=color, label=f"C = {C:g}")
ax.set(xlabel="markup, %", ylabel="predicted probability", title="With a penalty")
ax.legend(loc="upper right")
plt.show()

With a penalty the slope is finite: $-0.41$ at $C = 0.1$, $-1.65$ at $C = 1$ and $-3.96$ at $C = 10$. The curves are smooth again, and a flat next to the gap gets a probability near 0.5. The page `04_separable_and_penalty.html` has both parts with sliders.

The same penalty acts on the 40 flats, where no line splits the classes. Fit the two-feature model on standardised features for several values of $C$.

In [ ]:
c_rows = {}
print("         C   weight of markup   weight of metro   accuracy, 400 new   log-loss, 400 new")
for C in [0.001, 0.01, 0.1, 1, 10, np.inf]:
    fit = make_pipeline(StandardScaler(), LogisticRegression(C=C)).fit(X, sold)
    c_rows[C] = (
        *fit[-1].coef_[0],
        fit.score(X_new, sold_new),
        log_loss(sold_new, fit.predict_proba(X_new)[:, 1]),
    )
    label = "no penalty" if np.isinf(C) else f"{C:g}"
    print(
        f"{label:>10} {c_rows[C][0]:18.2f} {c_rows[C][1]:17.2f} {c_rows[C][2]:19.3f} "
        f"{c_rows[C][3]:19.3f}"
    )

At $C = 0.001$ the penalty wins: both weights are close to zero and the model answers "not sold" for every new flat, which is the baseline's 0.565. From $C = 0.1$ up the accuracy stays at 0.84, while the log-loss keeps falling until the weights are close to the ones without a penalty. These 40 flats have two features and overlapping classes, so they need little regularisation, unlike the model of the next section.

## 5. Curved boundaries

A new sample: 120 flats with an area in m² and a markup, and 400 new ones for testing. Mid-sized flats sell even with a markup, while very small and very large flats sell only at a discount. The data was generated with $\text{log-odds} = 3 - 0.2 \cdot \text{markup} - \bigl((\text{area} - 65) / 14\bigr)^2$, so the true boundary is a parabola.

In [ ]:
def area_sample(seed, n):
    """n flats: a matrix with the area in m² and the markup in percent, and sold (1) or not (0)."""
    rng = np.random.default_rng(seed)
    area = np.round(rng.uniform(30, 110, n), 0)
    markup = np.round(rng.uniform(-15, 25, n), 0)
    log_odds = 3.0 - 0.2 * markup - ((area - 65) / 14) ** 2
    sold = (rng.uniform(size=n) < 1 / (1 + np.exp(-log_odds))).astype(int)
    return np.column_stack([area, markup]), sold


X_area, sold_area = area_sample(2, 120)  # the 120 flats we train on
X_area_new, sold_area_new = area_sample(5002, 400)  # 400 new flats
print(f"{sold_area.sum()} of the {len(sold_area)} flats sold")

area_grid = np.linspace(28, 112, 300)
true_boundary = 5 * (3 - ((area_grid - 65) / 14) ** 2)  # the markup where the true log-odds are 0
fig, ax = plt.subplots(figsize=(7.5, 4.8))
scatter_sold(ax, X_area[:, 0], X_area[:, 1], sold_area, size=35)
ax.plot(area_grid, true_boundary, color=INK3, ls="--", lw=1.5, label="the true boundary")
ax.set(xlabel="area, m²", ylabel="markup, %", ylim=(-17, 27))
ax.legend(loc="lower left", bbox_to_anchor=(0, 1), ncols=3)
plt.show()

### 5.1 Polynomial features

No straight line separates these flats. As in week 3, the model can stay linear in the weights if it gets more features: all products of the two features up to a degree $d$.

In [ ]:
expand = PolynomialFeatures(2, include_bias=False).fit(X_area)  # no column of ones
print("degree 2:", ", ".join(expand.get_feature_names_out(["area", "markup"])))
print(
    "columns at degree 6:", PolynomialFeatures(6, include_bias=False).fit(X_area).n_output_features_
)


def tight_logistic(C):
    """Logistic regression that iterates until the weights stop moving."""
    return LogisticRegression(C=C, max_iter=1_000_000, tol=1e-10)

Degree 2 adds $\text{area}^2$, $\text{area} \cdot \text{markup}$ and $\text{markup}^2$, and the boundary becomes a second-order curve.

The helper `tight_logistic` sets a tight tolerance, `tol=1e-10`. With the default one the solver stops early on the degree 6 model below, and the numbers then depend on the version of the library.

**Exercise 5.1.** Write `area_model(degree, C)`. It returns a pipeline of four steps: `StandardScaler()`, `PolynomialFeatures(degree, include_bias=False)`, `StandardScaler()` again, because the powers come out on very different scales, and `tight_logistic(C)`.

In [ ]:
def area_model(degree, C):
    """Standardise, expand to all monomials up to the degree, standardise, logistic regression."""
    # YOUR CODE HERE: return make_pipeline(...) with the four steps
    return None

In [ ]:
model_2 = area_model(2, 1e6)
assert model_2 is not None, "Fill in area_model above"
model_2.fit(X_area, sold_area)
assert model_2[-1].coef_.shape == (1, 5), "Degree 2 should give 5 features: include_bias=False"
assert np.isclose(
    model_2.score(X_area, sold_area), 0.917, atol=0.005
), "Expect 0.917 on the 120 flats: scaler, polynomial features, scaler, logistic regression"
print(
    f"degree 2: {model_2[-1].coef_.shape[1]} features, "
    f"accuracy on the 120 flats {model_2.score(X_area, sold_area):.3f}"
)

Fit three degrees with almost no penalty, $C = 10^6$, and then degree 6 once more with the default $C = 1$.

In [ ]:
assert area_model(2, 1.0) is not None, "Finish Exercise 5.1 first"


def new_flats_loss(fit):
    """Log-loss of a fitted model on the 400 new flats. A probability of exactly 0 or 1 for a
    wrong answer would make it infinite, hence the same clip as in Exercise 2.2."""
    p = np.clip(fit.predict_proba(X_area_new)[:, 1], 1e-15, 1 - 1e-15)
    return log_loss(sold_area_new, p)


area_fits, area_rows = {}, {}
print(
    "degree       C   features   accuracy, 120 flats   accuracy, 400 new   log-loss, 400 new"
    "   largest |weight|"
)
for degree, C in [(1, 1e6), (2, 1e6), (6, 1e6), (6, 1.0)]:
    fit = area_model(degree, C).fit(X_area, sold_area)
    area_fits[degree, C] = fit
    area_rows[degree, C] = (
        fit[-1].coef_.shape[1],
        fit.score(X_area, sold_area),
        fit.score(X_area_new, sold_area_new),
        new_flats_loss(fit),
        np.abs(fit[-1].coef_).max(),
    )
    row = area_rows[degree, C]
    print(
        f"{degree:6d} {C:7g} {row[0]:10d} {row[1]:21.3f} {row[2]:19.3f} {row[3]:19.3f} "
        f"{row[4]:18.1f}"
    )

p_6 = area_fits[6, 1e6].predict_proba(X_area_new)[:, 1]
sure = (p_6 < 0.001) | (p_6 > 0.999)  # the model is all but certain
sure_wrong = sure & ((p_6 >= 0.5) != sold_area_new)
print(
    f"\ndegree 6, C = 1e+06: below 0.001 or above 0.999 for {sure.sum()} of the 400 new flats, "
    f"{sure_wrong.sum()} of these answers are wrong"
)

xx, yy, points = grid_points((28, 112), (-17, 27))  # x is the area, y is the markup
fig, axes = plt.subplots(1, 4, figsize=(16, 4.3), sharey=True)
for ax, ((degree, C), fit) in zip(axes, area_fits.items(), strict=True):
    p_grid = fit.predict_proba(points)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, p_grid, levels=np.linspace(0, 1, 11), cmap=PROB_CMAP, alpha=0.5)
    ax.contour(xx, yy, p_grid, levels=[0.5], colors=INK, linewidths=1.8)
    scatter_sold(ax, X_area[:, 0], X_area[:, 1], sold_area, size=22)
    ax.set(xlabel="area, m²", title=f"Degree {degree}, C = {'$10^6$' if C > 1 else '1'}")
axes[0].set_ylabel("markup, %")
fig.tight_layout()
plt.show()

Degree 1 underfits: a line cannot bend around the middle, and it gets 0.738 of the new flats right. Degree 2 has the right shape and gets 0.848. Degree 6 has 27 features for 120 flats. With almost no penalty it gets every training flat right, 1.000, and only 0.770 of the new ones. Its log-loss on the new flats is 4.37, where a model that always says 0.5 would score 0.693. The model is confident in its mistakes: it is all but certain about 347 of the 400 new flats, and 68 of those answers are wrong. Its largest weight is 1,052. With $C = 1$ the same 27 features get 0.843 of the new flats right, with a largest weight of 1.9.

### 5.2 Choosing C by cross-validation

$C$ is a hyperparameter, like $\lambda$ in week 3, and it is chosen the same way: by cross-validation on the training flats, with the whole pipeline refitted inside every fold. For a classifier the folds are stratified, so each keeps the same share of sold flats as the full sample. The score is the log-loss, `scoring="neg_log_loss"`, which is smoother than the accuracy on 24 validation flats.

**Exercise 5.2.** Build `search`: a `GridSearchCV` over `area_model(6, 1.0)` with the grid `{"logisticregression__C": C_GRID}`, `cv=folds` and `scoring="neg_log_loss"`. Fit it on `X_area`, `sold_area`.

In [ ]:
C_GRID = np.logspace(-3, 3, 13)  # 0.001 ... 1000, two values per decade
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
# YOUR CODE HERE: search = GridSearchCV(pipeline, grid, cv=..., scoring=...).fit(X_area, sold_area)
search = None

In [ ]:
assert search is not None, "Fill in search above"
assert hasattr(search, "best_params_"), "Fill in search above: call .fit(X_area, sold_area) on it"
assert area_model(2, 1.0) is not None, "Finish Exercise 5.1 first"
best_C = search.best_params_["logisticregression__C"]
assert np.isclose(
    best_C, 0.3162, atol=1e-4
), "Expect the best C 0.316: check the grid, cv and scoring"
cv_loss = -search.cv_results_["mean_test_score"]  # the mean log-loss over the folds, for every C
best_6 = search.best_estimator_  # already refitted on all 120 flats
best_6_scores = (best_6.score(X_area_new, sold_area_new), new_flats_loss(best_6))
print(f"degree 6: best C = {best_C:.3g}, cross-validation log-loss {cv_loss.min():.3f}")
print(f"on the 400 new flats: accuracy {best_6_scores[0]:.3f}, log-loss {best_6_scores[1]:.3f}")

# the same curve measured on the 400 new flats, which a real project does not have
new_loss = np.array([new_flats_loss(area_model(6, C).fit(X_area, sold_area)) for C in C_GRID])
print(
    f"the best C on the new flats: {C_GRID[new_loss.argmin()]:.3g}, log-loss {new_loss.min():.3f}"
)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(C_GRID, cv_loss, "o-", ms=4, color=ORANGE, label="5-fold cross-validation, 120 flats")
ax.plot(C_GRID, new_loss, "s-", ms=4, color=ACCENT, label="400 new flats")
ax.scatter([best_C], [cv_loss.min()], s=90, color=ORANGE, edgecolor="white", zorder=4)
ax.set(xscale="log", xlabel="C", ylabel="log-loss", ylim=(0.3, 1.0), title="Degree 6")
ax.legend(loc="upper center")
plt.show()

Cross-validation picks $C = 0.32$ with a log-loss of 0.376. Refitted on all 120 flats, this model gets 0.835 of the new flats right with a log-loss of 0.351. The orange curve needed only the training flats, and its minimum is one grid step away from the minimum of the curve for the new flats ($C = 1$, log-loss 0.347).

The degree is a hyperparameter too, so search over both at once.

In [ ]:
assert area_model(2, 1.0) is not None, "Finish Exercise 5.1 first"
assert search is not None, "Finish Exercise 5.2 first"
joint = GridSearchCV(
    area_model(6, 1.0),
    {"polynomialfeatures__degree": [1, 2, 3, 4, 5, 6], "logisticregression__C": C_GRID},
    cv=folds,
    scoring="neg_log_loss",
).fit(X_area, sold_area)
results = joint.cv_results_
degree_rows = {}
print("degree   best C   cross-validation log-loss   accuracy, 400 new   log-loss, 400 new")
for degree in range(1, 7):
    of_degree = np.nonzero(results["param_polynomialfeatures__degree"] == degree)[0]
    i = of_degree[results["mean_test_score"][of_degree].argmax()]  # the best C for this degree
    C = results["param_logisticregression__C"][i]
    fit = area_model(degree, C).fit(X_area, sold_area)
    degree_rows[degree] = (
        C,
        -results["mean_test_score"][i],
        fit.score(X_area_new, sold_area_new),
        new_flats_loss(fit),
    )
    row = degree_rows[degree]
    print(f"{degree:6d} {row[0]:8.3g} {row[1]:27.3f} {row[2]:19.3f} {row[3]:19.3f}")
best_degree = joint.best_params_["polynomialfeatures__degree"]
print(
    f"\nthe search picks degree {best_degree} with C = {joint.best_params_['logisticregression__C']:.3g}"
)

The search picks degree 2 with $C = 3.2$. Its cross-validation log-loss is 0.313, against 0.376 for the best model of degree 6, and on the new flats it gets 0.843 right with a log-loss of 0.335. So the penalty repairs degree 6, and the simpler model, the one that matches how the data was made, is still a little better. The gaps are small, though: with a tuned $C$ every degree from 2 to 6 gets between 0.835 and 0.845 of the new flats right, a difference of four flats out of 400.

## 6. More than two classes

### 6.1 One-vs-rest and softmax

Flats of three classes, economy, comfort and business, 50 of each, with an area and a price per m². Another 600 are for testing. The classes lie in a row with comfort in the middle.

In [ ]:
CLASS_NAMES = ["economy", "comfort", "business"]
CLASS_COLOURS = [ACCENT, ORANGE, GREEN]


def classes_sample(seed, n_per_class):
    """Flats of three classes: a matrix with the area in m² and the price per m² in US dollars,
    and the class number 0, 1 or 2."""
    rng = np.random.default_rng(seed)
    parts = []
    for label, (mean_area, mean_price) in enumerate([(45, 900), (65, 1300), (90, 1750)]):
        area = rng.normal(mean_area, 9, n_per_class)
        price_m2 = mean_price + 12 * (area - mean_area) + rng.normal(0, 130, n_per_class)
        labels = np.full(n_per_class, label)
        parts.append(np.column_stack([np.round(area), np.round(price_m2, -1), labels]))
    table = np.vstack(parts)
    return table[:, :2], table[:, 2].astype(int)


def scatter_classes(ax, X, y, size=30):
    """Flats of the three classes on the plane of area and price per m²."""
    for label, (name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLOURS, strict=True)):
        ax.scatter(
            X[y == label, 0],
            X[y == label, 1],
            s=size,
            color=color,
            edgecolor="white",
            linewidth=0.6,
            marker="os^"[label],
            zorder=3,
            label=name,
        )
    ax.set(xlabel="area, m²")


X_cls, y_cls = classes_sample(6, 50)  # 150 flats to train on
X_cls_new, y_cls_new = classes_sample(5006, 200)  # 600 new flats

fig, ax = plt.subplots(figsize=(7.5, 4.8))
scatter_classes(ax, X_cls, y_cls, size=40)
ax.set_ylabel("price per m², $")
ax.legend(loc="upper left")
plt.show()

A linear model can handle three classes in two ways. One-vs-rest trains three binary classifiers, each for one class against the other two, and answers with the class whose classifier gives the highest score. Softmax regression is a single model with a score $z_k = \mathbf{w}_k^\top \mathbf{x} + b_k$ for every class and the probabilities

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}.$$

It is trained by maximum likelihood, as in section 2, and its loss is called the cross-entropy. When the target has three or more classes, `LogisticRegression` fits softmax regression.

In [ ]:
# almost no penalty, C = 1e6, with the tight tolerance of section 5
ovr = make_pipeline(StandardScaler(), OneVsRestClassifier(tight_logistic(1e6))).fit(X_cls, y_cls)
softmax_model = make_pipeline(StandardScaler(), tight_logistic(1e6)).fit(X_cls, y_cls)
print("softmax regression, one row of weights per class (standardised features):")
print(softmax_model[-1].coef_.round(2))

**Exercise 6.1.** Write `softmax(scores)` for a matrix with one row per flat and one column per class. Every row of the result sums to 1.

In [ ]:
def softmax(scores):
    """Every row of scores turned into probabilities that sum to 1."""
    # YOUR CODE HERE: np.exp of the scores, divided by the sum of its row
    # (.sum(axis=1, keepdims=True) keeps the sums as a column, so the division works by rows)
    return None

In [ ]:
class_scores = softmax_model.decision_function(X_cls)  # the three scores z_k of every flat
probabilities = softmax(class_scores)
assert probabilities is not None, "Fill in softmax above"
assert probabilities.shape == (150, 3), "Expect one row per flat and one column per class"
assert np.allclose(probabilities.sum(axis=1), 1), "Every row should sum to 1"
assert np.allclose(
    probabilities, softmax_model.predict_proba(X_cls)
), "The result differs from predict_proba: exp of each score over the sum of exps in its row"
two_classes = softmax(np.array([[1.7, 0.0]]))
assert np.isclose(
    two_classes[0, 0], 1 / (1 + np.exp(-1.7))
), "Scores z and 0 should give sigmoid(z)"
print("class      the three scores          the three probabilities")
for i in [10, 60, 110]:  # one flat of each class
    print(
        f"{CLASS_NAMES[y_cls[i]]:9s}  {str(class_scores[i].round(2)):24s}  {probabilities[i].round(3)}"
    )
print(f"\ntwo classes with the scores 1.7 and 0: {two_classes[0].round(3)}")

With two classes and the scores $z$ and 0, softmax gives $e^{z} / (e^{z} + 1) = \sigma(z)$. Logistic regression is softmax regression for two classes.

Now compare the two methods on the 600 new flats. Next to the accuracy, count for each class the share of its flats that the model labels correctly.

In [ ]:
def class_scores_row(fit):
    """Accuracy on the 150 flats, on the 600 new ones, and the share right within each class."""
    predicted = fit.predict(X_cls_new)
    within = [np.mean(predicted[y_cls_new == label] == label) for label in range(3)]
    return fit.score(X_cls, y_cls), fit.score(X_cls_new, y_cls_new), *within


class_rows = {"one-vs-rest": class_scores_row(ovr), "softmax": class_scores_row(softmax_model)}
print("              accuracy              share right within a class, 600 new")
print("              150 flats   600 new   economy   comfort   business")
for name, row in class_rows.items():
    print(f"{name:12s} {row[0]:10.3f} {row[1]:9.3f} {row[2]:9.3f} {row[3]:9.3f} {row[4]:10.3f}")

# the three binary classifiers inside one-vs-rest, each on its own problem
Z_cls_new = ovr[0].transform(X_cls_new)
ovr_binary = {}
print("\none class against the rest      accuracy, 600 new   always 'the rest'")
for label, binary in enumerate(ovr[-1].estimators_):
    ovr_binary[CLASS_NAMES[label]] = (
        binary.score(Z_cls_new, y_cls_new == label),
        np.mean(y_cls_new != label),
    )
    print(
        f"{CLASS_NAMES[label]:28s} {ovr_binary[CLASS_NAMES[label]][0]:20.3f} "
        f"{ovr_binary[CLASS_NAMES[label]][1]:19.3f}"
    )

x_range = (X_cls_new[:, 0].min() - 3, X_cls_new[:, 0].max() + 3)
y_range = (X_cls_new[:, 1].min() - 60, X_cls_new[:, 1].max() + 60)
xx, yy, points = grid_points(x_range, y_range)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, (name, fit) in zip(
    axes, [("One-vs-rest", ovr), ("Softmax regression", softmax_model)], strict=True
):
    regions = fit.predict(points).reshape(xx.shape)
    ax.contourf(xx, yy, regions, levels=[-0.5, 0.5, 1.5, 2.5], colors=CLASS_COLOURS, alpha=0.18)
    scatter_classes(ax, X_cls_new, y_cls_new, size=14)
    ax.set_title(f"{name}, 600 new flats")
axes[0].set_ylabel("price per m², $")
axes[0].legend(loc="upper left")
fig.tight_layout()
plt.show()

Softmax regression gets 0.872 of the new flats right and one-vs-rest 0.838. The difference is in the middle class: one-vs-rest labels 0.635 of the comfort flats correctly, and softmax 0.810. The three binary problems show why. A line separates economy from the rest and business from the rest, but no line cuts the middle class from both of its neighbours. The classifier "comfort against the rest" is right for 0.653 of the new flats, which is less than answering "the rest" every time would get (0.667). Its weak scores lose to the other two classifiers, and the comfort region shrinks. Softmax regression compares three scores that were trained together. The page `05_multiclass.html` shows the three boundaries one at a time.

### 6.2 Real data: tumours

569 breast tumours with 30 measurements of the cell nuclei each, bundled with scikit-learn. The target is 1 for a benign tumour and 0 for a malignant one. A quarter of the data is held out as the test set, stratified by the target as in week 3.

In [ ]:
cancer = load_breast_cancer()  # bundled with scikit-learn, no download
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    cancer.data, cancer.target, test_size=0.25, stratify=cancer.target, random_state=SEED
)
print(
    f"{len(cancer.target)} tumours, {cancer.data.shape[1]} features: "
    f"{len(y_c_train)} to train on, {len(y_c_test)} to test on"
)
print(f"benign: {y_c_train.mean():.3f} of the training set, {y_c_test.mean():.3f} of the test set")
print("\nthree of the features          smallest    largest")
for name in ["mean radius", "mean area", "mean smoothness"]:
    column = cancer.data[:, list(cancer.feature_names).index(name)]
    print(f"{name:28s} {column.min():10.3f} {column.max():10.3f}")

The features live on very different scales, from hundredths to thousands, so the scaler goes into the pipeline.

**Exercise 6.2.** Build `cancer_model`: `make_pipeline(StandardScaler(), LogisticRegression())`, fitted on the training set.

In [ ]:
# YOUR CODE HERE: cancer_model = make_pipeline(...).fit(X_c_train, y_c_train)
cancer_model = None

In [ ]:
assert cancer_model is not None, "Fill in cancer_model above"
assert "standardscaler" in getattr(
    cancer_model, "named_steps", {}
), "Put StandardScaler() in front of the model with make_pipeline"
cancer_pred = cancer_model.predict(X_c_test)
cancer_acc = np.mean(cancer_pred == y_c_test)
cancer_loss = log_loss(y_c_test, cancer_model.predict_proba(X_c_test)[:, 1])
assert cancer_acc > 0.95, "Expect a test accuracy above 0.95: fit on X_c_train, y_c_train"
print(f'always "benign":      accuracy {y_c_test.mean():.3f}')
print(
    f"logistic regression: accuracy {cancer_acc:.3f} "
    f"({np.sum(cancer_pred == y_c_test)} of {len(y_c_test)}), log-loss {cancer_loss:.3f}"
)
print(
    f"benign called malignant: {np.sum((y_c_test == 1) & (cancer_pred == 0))}, "
    f"malignant called benign: {np.sum((y_c_test == 0) & (cancer_pred == 1))}"
)

The model is right about 141 of the 143 test tumours, an accuracy of 0.986, where always answering "benign" gets 0.629. Its log-loss is 0.068. Another split would move the count of mistakes by a few. One of the two mistakes is a benign tumour called malignant and the other is a malignant tumour called benign. Accuracy counts them the same, although the second one costs far more.

Now read the weights. The features were standardised, so the weights can be compared with each other: each one is the change in the log-odds of "benign" per standard deviation of its feature.

In [ ]:
assert cancer_model is not None, "Finish Exercise 6.2 first"
cancer_w = cancer_model[-1].coef_[0]
order = np.argsort(-np.abs(cancer_w))  # from the largest |weight| down
print("feature                  weight   multiplies the odds of benign by")
for j in order[:6]:
    print(f"{cancer.feature_names[j]:22s} {cancer_w[j]:8.2f} {np.exp(cancer_w[j]):12.2f}")

top = order[:10][::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(
    cancer.feature_names[top],
    cancer_w[top],
    color=[ACCENT if w > 0 else ORANGE for w in cancer_w[top]],
)
ax.axvline(0, color=LINE, lw=0.8)
ax.set(xlabel="weight of the standardised feature", title="The ten largest weights")
plt.show()

radius, perimeter = cancer.data[:, 0], cancer.data[:, 2]  # mean radius and mean perimeter
twins = np.corrcoef(radius, perimeter)[0, 1]
print(f"correlation of mean radius and mean perimeter: {twins:.3f}")

The six largest weights are all negative: a tumour with a rougher texture, a larger area or a more concave outline is less likely to be benign. One standard deviation more of `worst texture` multiplies the odds of benign by 0.29. Read single weights with care, as in week 3. Many of these features are near twins (the correlation of `mean radius` and `mean perimeter` is 0.998), and the penalty shares the weight between them.

### 6.3 Real data: handwritten digits

1,797 images of handwritten digits, 8 by 8 pixels each, also bundled with scikit-learn. Every pixel is a feature, so an image is a row of 64 numbers, and there are ten classes.

In [ ]:
digits = load_digits()  # bundled with scikit-learn, no download
X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    digits.data, digits.target, test_size=0.25, stratify=digits.target, random_state=SEED
)
print(
    f"{len(digits.target)} images, {digits.data.shape[1]} features: "
    f"{len(y_d_train)} to train on, {len(y_d_test)} to test on"
)

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, image, label in zip(axes, X_d_train, y_d_train, strict=False):
    ax.imshow(image.reshape(8, 8), cmap="gray_r")
    ax.set_title(str(label), fontsize=11, loc="center")
    ax.axis("off")
plt.show()

**Exercise 6.3.** Build `digit_model`: the same pipeline as in Exercise 6.2, fitted on `X_d_train`, `y_d_train`. With ten classes it is softmax regression.

In [ ]:
# YOUR CODE HERE: digit_model = make_pipeline(...).fit(X_d_train, y_d_train)
digit_model = None

In [ ]:
assert digit_model is not None, "Fill in digit_model above"
digit_pred = digit_model.predict(X_d_test)
digit_acc = np.mean(digit_pred == y_d_test)
assert digit_acc > 0.95, "Expect a test accuracy above 0.95: scaler, then LogisticRegression()"
confidence = digit_model.predict_proba(X_d_test).max(axis=1)  # the probability of the answer
wrong = np.nonzero(digit_pred != y_d_test)[0]
right = digit_pred == y_d_test
print(f"accuracy on the test images: {digit_acc:.3f} ({right.sum()} of {len(y_d_test)})")
print(
    f"mean probability of the answer: {confidence[right].mean():.2f} when it is right, "
    f"{confidence[wrong].mean():.2f} when it is wrong"
)

shown = wrong[:10]
fig, axes = plt.subplots(1, len(shown), figsize=(1.25 * len(shown), 2.1), squeeze=False)
for ax, i in zip(axes[0], shown, strict=True):
    ax.imshow(X_d_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(
        f"{y_d_test[i]} read as {digit_pred[i]}\n({confidence[i]:.2f})",
        fontsize=10,
        loc="center",
        fontweight="normal",
    )
    ax.axis("off")
fig.tight_layout()
plt.show()

A linear model on raw pixels reads 0.978 of the test digits correctly, 440 of 450, where a random guess gets about 0.10. Above are its 10 mistakes, with the probability it gave to its answer in brackets. Some are hard for a person too. The model was less sure about them: its answer got a probability of 0.67 on average, against 0.96 for the digits it read correctly.

## 7. What to remember

- Least squares on the labels 0 and 1 gives answers outside the range from 0 to 1 and bends towards points that were already on the right side.
- Logistic regression models the log-odds with a linear function. The probability is the sigmoid of the score, the loss is the log-loss, and its gradient is $\frac{1}{n} X^\top (\mathbf{p} - \mathbf{y})$. The minimum has no closed form, but the loss is convex and gradient descent finds it.
- A weight is the change in the log-odds per unit of its feature, and $e^{w}$ multiplies the odds. The boundary $p = 0.5$ is a straight line.
- When a line splits the classes perfectly, the weights grow without limit, and the L2 penalty stops them. In scikit-learn a small `C` is a strong penalty. Choose `C` by cross-validation on stratified folds with the whole pipeline inside.
- Polynomial features bend the boundary. For several classes, softmax regression is one model with a score for every class.
- Measure accuracy and log-loss on data the model has not seen, and compare them with a baseline.

## Try at home

1. In section 2.4, try $\eta = 8$, 12 and 15. Which is the fastest, and where does the descent stop converging?
2. Run `gradient_descent` on the raw features, `np.column_stack([np.ones(40), X])`, with $\eta = 0.01$ for 20,000 steps. How many of them does it take to get within 0.001 of the minimum, and what does that say about standardising?
3. In section 6.2, remove the scaler from the pipeline. What does scikit-learn report, and what happens to the test accuracy?
4. In section 6.3, fit `KNeighborsClassifier` from week 1 on the same split and compare its accuracy and its mistakes with those of softmax regression.